In [1]:
import argparse
import time
import numpy as np
import wandb

from reasoning.tools.utils import load_model_with_vllm
from datasets import load_dataset
from vllm import SamplingParams
from reasoning.inference.majority import MajorityInference
from reasoning.tools.logger import load_config, apply_config
import numpy as np
import argparse

from reasoning.evaluator.math_grader import math_equal, extract_answer

from reasoning.models.model import ValueModel_qwen

In [2]:
# Does not support batch inference

config = "/ssdscratch/byuan48/efficient_reasoning/configs/samplingTree_llama_16_version1.yaml"
config = load_config(config)
config['sampling_method'] = 'stochastic_beam_search_version3'
apply_config(config) 
print(config)

# load dataset and model
dataset = load_dataset(config["data_path"])
dataset = dataset['test']
model_name = config["model_name"]

policy_model, tokenizer = load_model_with_vllm(model_name, task='auto', tensor_parallel_size=len(config["cuda_device_ids"]), gpu_memory_utilization=0.8)
tokenizer.pad_token = tokenizer.eos_token 
reward_model = config.get("reward_model", None)
reward_model = ValueModel_qwen(device = "auto")

# inference hyperparameters
num_return_sequences = config["num_return_sequences"]
assert num_return_sequences == 1
max_new_tokens = config["max_new_tokens"]
temperature = config["temperature"]
top_p = config["top_p"]
system_prompt = config["system_prompt"]

# SamplingTree parameters
sampling_method = config["sampling_method"]
samplingTree_temperature = config["samplingTree_temperature"]
beam_width = config["beam_width"]
max_steps = config["max_steps"]

# name of the results file
config_name = config["config_name"]

# Also set the seed for the sampling params 
sampling_params = SamplingParams(
    temperature=temperature,
    max_tokens=max_new_tokens,
    n=num_return_sequences,
    top_p=top_p,
    stop_token_ids=[tokenizer.eos_token_id],
    skip_special_tokens = True,
    include_stop_str_in_output = False,
) # shouldn't set seed for random sampling


wandb: Currently logged in as: yuanbo096. Use `wandb login --relogin` to force relogin
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.


{'config_name': 'Llama-3.2-1B-Instruct-samplingTree_16_version1', 'batch_size': 1, 'cuda_device_ids': [0, 1, 2, 3], 'seed': 42, 'data_path': 'HuggingFaceH4/MATH-500', 'model_name': 'meta-llama/Llama-3.2-1B-Instruct', 'system_prompt': 'Solve the following reasoning problem, strictly following the instructions below:\n1. Show your reasoning step by step. Each step should start with "## Step X:"\n2. Put your final answer within \\boxed{} at the end\n', 'max_new_tokens': 2048, 'temperature': 0.7, 'top_p': 0.9, 'experiment_name': 'samplingTree_experiment', 'inference_method': 'samplingTree', 'num_return_sequences': 1, 'wandb_project': 'efficient_reasoning', 'sampling_method': 'stochastic_beam_search_version3', 'samplingTree_temperature': 0.1, 'beam_width': 16, 'max_steps': 2, 'threshold': 0.9}
INFO 04-15 21:15:17 config.py:510] This model supports multiple tasks: {'reward', 'classify', 'generate', 'score', 'embed'}. Defaulting to 'generate'.
INFO 04-15 21:15:17 config.py:1310] Defaulting to

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


(VllmWorkerProcess pid=2409129) INFO 04-15 21:15:21 weight_utils.py:296] No model.safetensors.index.json found in remote.
(VllmWorkerProcess pid=2409131) INFO 04-15 21:15:21 weight_utils.py:296] No model.safetensors.index.json found in remote.
INFO 04-15 21:15:21 model_runner.py:1099] Loading model weights took 0.5968 GB
(VllmWorkerProcess pid=2409130) INFO 04-15 21:15:21 weight_utils.py:296] No model.safetensors.index.json found in remote.
(VllmWorkerProcess pid=2409129) INFO 04-15 21:15:21 model_runner.py:1099] Loading model weights took 0.5968 GB
(VllmWorkerProcess pid=2409131) INFO 04-15 21:15:22 model_runner.py:1099] Loading model weights took 0.5968 GB
(VllmWorkerProcess pid=2409130) INFO 04-15 21:15:22 model_runner.py:1099] Loading model weights took 0.5968 GB
(VllmWorkerProcess pid=2409129) INFO 04-15 21:15:25 worker.py:241] Memory profiling takes 3.17 seconds
(VllmWorkerProcess pid=2409129) INFO 04-15 21:15:25 worker.py:241] the current vLLM instance can use total_gpu_memory (

Capturing CUDA graph shapes:  97%|█████████▋| 34/35 [00:18<00:00,  1.92it/s]

(VllmWorkerProcess pid=2409131) INFO 04-15 21:15:49 model_runner.py:1535] Graph capturing finished in 20 secs, took 0.24 GiB


Capturing CUDA graph shapes: 100%|██████████| 35/35 [00:19<00:00,  1.77it/s]

INFO 04-15 21:15:49 model_runner.py:1535] Graph capturing finished in 20 secs, took 0.24 GiB
(VllmWorkerProcess pid=2409130) INFO 04-15 21:15:49 model_runner.py:1535] Graph capturing finished in 20 secs, took 0.24 GiB
(VllmWorkerProcess pid=2409129) INFO 04-15 21:15:49 model_runner.py:1535] Graph capturing finished in 20 secs, took 0.24 GiB
INFO 04-15 21:15:49 llm_engine.py:431] init engine (profile, create kv cache, warmup model) took 27.12 seconds


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Some weights of the model checkpoint at Qwen/Qwen2.5-Math-PRM-7B were not used when initializing Qwen2ForProcessRewardModel: ['lm_head.weight']
- This IS expected if you are initializing Qwen2ForProcessRewardModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing Qwen2ForProcessRewardModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [3]:
import importlib
import reasoning.inference.tree
tree_module = importlib.reload(reasoning.inference.tree)
Tree = tree_module.Tree
config['sampling_method'] = 'stochastic_beam_search_version4'
sampling_method = config['sampling_method']
# random sampling from dataset
dataset = dataset.shuffle(seed=42).select(range(100))

In [ ]:
start_time = time.time()
threshold = 0.9
right_count = 0
num_generated_tokens = 0
for i in range(0, len(dataset)):
    question = dataset['problem'][i]
    answer = dataset['solution'][i]        

    tree = Tree(system_prompt, question, policy_model, reward_model, sampling_method, config_name, sampling_params, samplingTree_temperature, beam_width, threshold)
    for _ in range(max_steps):
        path = tree.generate_next_trajectory() 
        tree.paths.append(path)
    majority_inference = MajorityInference(policy_model=policy_model, tokenizer=tokenizer, sampling_params=sampling_params, config_name=config_name, reward_model=reward_model, method = 'weighted_majority')
    all_solutions  = [path.solutions for path in tree.explored_paths]
    rewards = [np.min(path.scores) for path in tree.explored_paths]
    
    solution = majority_inference.weighted_majority_vote(all_solutions, rewards)   
    extracted_answer = extract_answer(answer)
    is_correct = math_equal(extracted_answer, solution)
    if extracted_answer is None:
        raise ValueError('extracted answer is None')
    if is_correct:
        right_count += 1
    num_generated_tokens += tree.num_generated_tokens
    print("--------------------------------")
    print(f'question: {question}')
    print(f'solution: {solution}')
    print(f'extracted answer: {extracted_answer}')
    print(f'is correct: {is_correct}')
    print(f'Accuracy: {right_count/(i+1)}')
    print(f'processed {i+1} questions')
    print(f'Number of generated tokens: {tree.num_generated_tokens}')
    print("--------------------------------")


wandb.log({"Accuracy": right_count/len(dataset)})
wandb.log({"Number of generated tokens": num_generated_tokens/len(dataset)})
end_time = time.time()
print("Time taken: {} seconds".format(end_time - start_time))

wandb.finish()

INFO 04-15 21:15:58 chat_utils.py:333] Detected the chat template content format to be 'string'. You can set `--chat-template-content-format` to override this.


Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts: 100%|██████████| 1/1 [00:01<00:00,  1.26s/it, est. speed input: 247.18 toks/s, output: 265.46 toks/s]
We detected that you are passing `past_key_values` as a tuple and this is deprecated and will be removed in v4.43. Please use an appropriate `Cache` class (https://huggingface.co/docs/transformers/v4.41.3/en/internal/generation_utils#transformers.Cache)
Processed prompts: 100%|██████████| 16/16 [00:08<00:00,  1.94it/s, est. speed input: 767.63 toks/s, output: 1137.11 toks/s] 


found solutions: ['4', '-6', '4', '3', '1', '-4', None, '8', '10', '3', '6', '4', '8', '12', '11', '10']
rewards: [0.006011962890625, 0.01324462890625, 0.01397705078125, 0.038818359375, 0.00482177734375, 0.0279541015625, 0.03564453125, 0.0079345703125, 0.0079345703125, 0.0255126953125, 0.017578125, 0.034423828125, 0.06884765625, 0.01361083984375, 0.0205078125, 0.025146484375]
--------------------------------
question: Below is a magic square, meaning that the sum of the numbers in each row, in each column, and in each of the $2$ main diagonals are equal. What is the value of $n$?

[asy]size(125);
for(int i = 0; i<4; ++i)
{

draw((0,i)--(3,i),linewidth(1));
}

for(int j = 0; j<4; ++j)
{

draw((j,0)--(j,3),linewidth(1));
}

label("$n-3$",(.5,.5));
label("3",(.5,1.5));
label("$n+1$",(.5,2.5));

label("$n+2$",(1.5,.5));
label("$2n-9$",(1.5,1.5));
label("$1$",(1.5,2.5));

label("$2$",(2.5,.5));
label("$n$",(2.5,1.5));
label("$n-1$",(2.5,2.5));
[/asy]
solution: 8
extracted answer: 7
is corre

Processed prompts: 100%|██████████| 1/1 [00:00<00:00,  1.37it/s, est. speed input: 172.86 toks/s, output: 292.22 toks/s]


...

found solutions: ['5', '4', None, '2', '4', '4', '5', '2', '2', '1', '4', '4', None, None, '2', '3']
rewards: [0.005645751953125, 0.9921875, 0.9921875, 0.002471923828125, 0.65625, 0.02978515625, 0.00860595703125, 0.00154876708984375, 0.00860595703125, 0.0206298828125, 0.0908203125, 0.9921875, 0.04931640625, 0.004547119140625, 0.01153564453125, 0.0189208984375]
--------------------------------
question: A polynomial with integer coefficients is of the form
\[2x^4 + a_3 x^3 + a_2 x^2 + a_1 x + 1 = 0.\]Find the number of different possible rational roots of this polynomial.
solution: 4
extracted answer: 4
is correct: True
Accuracy: 0.46464646464646464
processed 99 questions
Number of generated tokens: 9143
--------------------------------


Processed prompts: 100%|██████████| 16/16 [00:10<00:00,  1.51it/s, est. speed input: 392.23 toks/s, output: 2211.05 toks/s]


found solutions: [None, None, '\\frac{1}{2}', '\\frac{1}{3}', None, None, None, '1', None, '2', None, '1', '0', '\\frac{1}{1}', None, '\\frac{1}{2}']
rewards: [0.00872802734375, 0.05419921875, 0.10302734375, 0.005218505859375, 0.08056640625, 0.0172119140625, 0.010986328125, 0.01953125, 0.0140380859375, 0.01708984375, 0.032470703125, 0.035888671875, 0.01153564453125, 0.10986328125, 0.0135498046875, 0.01031494140625]
--------------------------------
question: Compute
\[\sum_{n = 1}^\infty \frac{F_{n + 1}}{F_n F_{n + 2}},\]where $F_n$ denotes the $n$th Fibonacci number, so $F_0 = 0$ and $F_1 = 1.$
solution: 1
extracted answer: 2
is correct: False
Accuracy: 0.46
processed 100 questions
Number of generated tokens: 24544
--------------------------------
Time taken: 1144.3021311759949 seconds


Accuracy,▁
Number of generated tokens,▁
Accuracy,0.46
Number of generated tokens,10527.4


: 